# Classical simulation

Adapted from Qualtran v0.7.0, Copyright 2023 Google LLC, Apache-2.0. [Locked source](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/simulation/classical_sim.ipynb).

Reversible arithmetic can be inspected on basis values before checking its full quantum action. This notebook uses FlagQuantum-owned numerical callbacks and real stored component bodies. The quantum user interface remains `circuit.append(op)`. Run from the repository root with the optional numerical and drawing dependencies installed.

In [ ]:
from examples.algorithms.classical_simulation import cnot_values

cnot_result = tuple(cnot_values({"ctrl": 1, "target": 0}).values())
cnot_result

## Interface

Numerical callbacks accept named register values and return named outputs. Signature order fixes the returned tuple order. Stored component execution uses the existing Core program and classical state owner; foreign graph objects are unnecessary.

In [ ]:
import itertools

cnot_rows = []
for c, t in itertools.product([0, 1], repeat=2):
    out_c, out_t = cnot_values({"ctrl": c, "target": t}).values()
    cnot_rows.append(((c, t), (out_c, out_t)))
    print(f"{c}{t} -> {out_c}{out_t}")

# The same native operation remains appendable through the ordinary circuit API.
import torch
import flagquantum as fq
from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.classical_tensor_crosscheck import crosscheck_program
from flagquantum.compiler.registers.register_binding import bind_register_program

cnot_body = bind_register_program(crosscheck_program("cnot"), ((0,), (1,)))
initial = torch.tensor([0, 0, 1, 0], dtype=torch.complex128)
circuit = fq.Circuit(2, inputs=initial, dtype=torch.complex128)
circuit.append(FieldOperation(2, cnot_body.circuit.instructions))
torch.testing.assert_close(circuit.state().flatten(), torch.tensor([0, 0, 0, 1], dtype=torch.complex128))


## Additional functionality

### Consistent classical action

Compare the direct word-addition callback with the actual ripple-carry graph. The graph calls the shared CNOT, AND, checked erasure and split/join numerical owners; it is not replaced by a whole-word addition callback.

In [ ]:
from examples.algorithms.classical_simulation import addition_values, addition_state

add_result = tuple(addition_values(8, {"a": 5, "b": 7}).values())
add_result

The following ten input pairs use the original seed 52 and original eight-bit range. Carries are computed and checked before numerical erasure. This numerical check is separate from physical measurements and their complex branch amplitudes.

In [ ]:
import numpy as np

rs = np.random.RandomState(seed=52)
sampled_pairs = []
for _ in range(10):
    a, b = rs.randint(256), rs.randint(256)
    ref_vals = addition_values(8, {"a": a, "b": b})
    decomp_vals = addition_state(8, {"a": a, "b": b}).simulate()
    assert ref_vals == decomp_vals
    sampled_pairs.append(((a, b), tuple(decomp_vals.values())))
    print("✓", end="")

### Exhaustive small-domain verification

Enumerate both two-bit input domains and compare the direct callback with the stored decomposition. This covers all 16 inputs, rather than sampling.

In [ ]:
exhaustive_rows = []
for a, b in itertools.product(range(4), repeat=2):
    expected = addition_values(2, {"a": a, "b": b})
    actual = addition_state(2, {"a": a, "b": b}).simulate()
    assert actual == expected
    exhaustive_rows.append(((a, b), tuple(actual.values())))
print("✓")

### Printing truth tables

Existing domain enumeration and formatting preserve input/output signature order and the original table separator.

In [ ]:
from examples.algorithms.classical_simulation import addition_table
from flagquantum.simulation.classical_truth_table import format_classical_truth_table

add_table = addition_table(2)
add_text = format_classical_truth_table(*add_table)
print(add_text)

## Implementation

A classical action is supplied explicitly. High-level numerical actions provide useful independent references for their decompositions. The next function demonstrates controlled NOT without a foreign base class or a provider registry.

In [ ]:
from flagquantum.core.ir.program.registers import Register, RegisterType
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.simulation.bit_encoding import bit_value_domain
from flagquantum.simulation.classical_truth_table import classical_truth_table

In [ ]:
def cnot_example(values: dict[str, object]) -> tuple[int, int]:
    control, target = values["ctrl"], values["target"]
    assert isinstance(control, int) and isinstance(target, int)
    return control, (control + target) % 2

Inspect the truth table of the explicitly supplied action. The same four inputs also match the shared native CNOT value owner.

In [ ]:
bit = RegisterType("bit", 1)
cnot_signature = Signature([Register("ctrl", bit), Register("target", bit)])
custom_table = classical_truth_table(cnot_signature, lambda reg: bit_value_domain(), cnot_example)
custom_text = format_classical_truth_table(*custom_table)
assert custom_table[2] == cnot_rows
print(custom_text)

## Properties and relations

### Inspecting classical values

The numerical state retains assignments after every actual call. A native call/value table shows the same arithmetic computation and checked carry erasure. This is a Matplotlib presentation rather than a Graphviz SVG layout replica; SSA call ordinals replace foreign dangle identities.

In [ ]:
from examples.algorithms.classical_simulation import addition_trace, draw_addition_trace

trace = addition_trace(2, {"a": 3, "b": 2})
assert addition_state(2, {"a": 3, "b": 2}).simulate() == {"a": 3, "b": 1}
trace_figure = draw_addition_trace(2, {"a": 3, "b": 2})
trace_figure

### Register bit encodings

Native integer codecs use big-endian order: index zero contains the most significant bit. Encoding 254 on eight bits yields seven ones followed by zero.

In [ ]:
from flagquantum.simulation.integer_encoding import integer_to_bits

encoded_254 = integer_to_bits(254, 8)
assert encoded_254 == [1, 1, 1, 1, 1, 1, 1, 0]
encoded_254

### Fixed-point stored integers

A four-bit fractional word stores an integer scaled by 16. This example starts with the exactly representable value 0.25 (stored integer 4), doubles modulo 16 and decodes to 0.5. Exact rational scaling is explicit here; arbitrary float quantization, rounding and overflow policies are not introduced or claimed. Doubling discards overflow and is therefore a numerical action, not an in-place quantum permutation.

In [ ]:
from fractions import Fraction
from flagquantum.simulation.fixed_point_encoding import fixed_point_real_value, validate_fixed_point_value

num_frac = 4
x_float = 0.25
scaled = Fraction(x_float) * 2**num_frac
assert scaled.denominator == 1
x_as_int = int(scaled)
validate_fixed_point_value(x_as_int, 4, num_frac)
x_out_as_int = (x_as_int * 2) % 2**num_frac
validate_fixed_point_value(x_out_as_int, 4, num_frac)
x_out_float = fixed_point_real_value(x_out_as_int, 4, num_frac)
assert x_out_float == 0.5
x_out_float